# Apriori on the Poilu & Co orders
Copy of the course notebook **Apriori_e_commerce.ipynb**, adapted to run in VS Code on **our orders**.

| Part | Lecture |
|---|---|
| 1. Course code on our orders (1 basket = 1 order) | Association Rule Mining: support, confidence, lift |
| 2. Check on the lecture examples (Figures 1, 2, 3) | Association Rule Mining |
| 3. Apriori step by step (pseudo-code of Figures 4 and 5) on our orders | Association Rule Mining: Apriori algorithm |
| 4. *Fréquemment achetés ensemble* on the product page | Recommender Systems 1.1 ("Customers who bought this also bought") |
| 5. *Recommandé pour vous* on the account page | Recommender Systems 1.7: Association Rule Mining for CF |
| 6. Precision and recall | Recommender Systems 1.10: Evaluation |

What changed compared to the course notebook (everything else is the course code):
- the Google Colab cells (`files.upload()`, `/content/...`, `!pip install`) and the Python tutorial cells (`sorted`, `lambda`, `map`) are removed;
- instead of `store_data.csv` we use our orders. The new functions are in `apriori_analysis.py` and `apriori_lecture.py` (same folder);
- the thresholds of `apriori()` are adapted to our data (~170 orders instead of 7501 baskets);
- the results are saved in `output/apriori_rules.json`, read by the website.

**Before running:** start MySQL in XAMPP (otherwise the demo data of `database/database.sql` is used).

# 1. Course code on our orders
## Installing necessary library
The libraries (`apyori`, `pandas`, `pymysql`…) are installed with `requirements.txt` at the root of the project (see the README, section "Python environment"). In Colab it was `!pip install apyori`.

## Import libraries

In [2]:
import numpy as np
import pandas as pd
from apyori import apriori

# NEW: our functions (see apriori_analysis.py and apriori_lecture.py)
from apriori_analysis import *
from apriori_lecture import apriori_lecture, candidate_gen, generate_rules, support

## Load dataset
Instead of `pd.read_csv('/content/store_data.csv', header=None)`:
1. `load_order_items()` reads one row per product of each order (cancelled orders are left out);
2. `orders_to_store_data()` builds the **same table as store_data.csv**: 1 row = 1 basket (= 1 order = 1 *transaction*), 1 product (*item*) per column, `NaN` in the empty cells.

In [3]:
order_items = load_order_items()
order_items.head()

Orders read from the MySQL database "poilu_and_co".


,order_id,customer,product_id,product_name
0,1,4,5,Médaille ronde en laiton
1,1,4,2,Collier cuir brodé au prénom
2,2,guest-2,23,Coffret anniversaire
3,3,6,21,Coffret adoption chiot
4,4,11,7,Médaille poisson pour chat


In [4]:
store_data = orders_to_store_data(order_items)
print(store_data.head())

                            0                               1    2
0    Médaille ronde en laiton    Collier cuir brodé au prénom  NaN
1        Coffret anniversaire                             NaN  NaN
2      Coffret adoption chiot                             NaN  NaN
3  Médaille poisson pour chat  Collier chat avec grelot gravé  NaN
4    Médaille ronde en laiton    Collier cuir brodé au prénom  NaN


## Filter out 'nan' values
Same loop as the course. We use `range(len(store_data))` (the alternative written in the course comment) because we do not have 7501 baskets. `records` is the *transaction database T* of the lecture.

In [5]:
records = []
for i in range(len(store_data)): #OR for i in range(0, 7501) in the course
    records.append([str(item) for item in store_data.values[i] if str(item) != 'nan'])

In [6]:
records[:2]

[['Médaille ronde en laiton', 'Collier cuir brodé au prénom'],
 ['Coffret anniversaire']]

In [7]:
len(records)   # n = number of transactions

172

## Define and run the algorithm
Thresholds adapted to our data (set in `apriori_analysis.py`):
- `min_support = 0.02` (*minsup*): the products are together in at least ~4 of our ~170 orders (the course used 0.0045 on 7501 baskets);
- `min_confidence = 0.2` (*minconf*): same as the course;
- `min_lift = 1.5`: the products are bought together at least 1.5 times more often than under independence (the course used 3 on a big supermarket dataset).

Choosing minsup is the weakness of Apriori named in the lecture: too high and rare products are missed, too low and the number of itemsets explodes.

In [8]:
association_rules = apriori(records, min_support=MIN_SUPPORT, min_confidence=MIN_CONFIDENCE, min_lift=MIN_LIFT, min_length=2)
association_results = list(association_rules)
association_results

[RelationRecord(items=frozenset({'Médaille poisson pour chat', 'Collier chat avec grelot gravé'}), support=0.046511627906976744, ordered_statistics=[OrderedStatistic(items_base=frozenset({'Collier chat avec grelot gravé'}), items_add=frozenset({'Médaille poisson pour chat'}), confidence=0.5333333333333333, lift=8.33939393939394), OrderedStatistic(items_base=frozenset({'Médaille poisson pour chat'}), items_add=frozenset({'Collier chat avec grelot gravé'}), confidence=0.7272727272727272, lift=8.339393939393938)]),
 RelationRecord(items=frozenset({'Gamelle inox gravée', 'Collier cuir brodé au prénom'}), support=0.029069767441860465, ordered_statistics=[OrderedStatistic(items_base=frozenset({'Collier cuir brodé au prénom'}), items_add=frozenset({'Gamelle inox gravée'}), confidence=0.20833333333333334, lift=4.479166666666667), OrderedStatistic(items_base=frozenset({'Gamelle inox gravée'}), items_add=frozenset({'Collier cuir brodé au prénom'}), confidence=0.625, lift=4.479166666666667)]),
 R

Support         = freq(A,B)/N

Confidence = freq(A,B)/freq(A)         -> How often A, B occur together, given number of times A occur

Lift = support/supp(A).supp(B)           -> independece occurance of A,B

Same measures with the notation of the lecture, for a rule $X \Rightarrow Y$ (antecedent $X$, consequent $Y$):

- $sup(X \Rightarrow Y) = P(X \cup Y) = \frac{\#(X \cup Y)}{n}$
- $conf(X \Rightarrow Y) = \frac{sup(X \cup Y)}{sup(X)} = P(Y|X)$
- $lift(X \Rightarrow Y) = \frac{sup(X \cup Y)}{sup(X)\,sup(Y)}$ (> 1: more often together than under independence)

## Sorting results based on their confidence values

In [9]:
listRules = sorted(
    [list(rule.ordered_statistics) for rule in association_results],
    key=lambda x: x[0].confidence,
    reverse=True
)

In [10]:
listRules[0]

[OrderedStatistic(items_base=frozenset({'Coussin portrait'}), items_add=frozenset({'Plaid portrait de votre animal'}), confidence=0.8461538461538463, lift=13.230769230769232),
 OrderedStatistic(items_base=frozenset({'Plaid portrait de votre animal'}), items_add=frozenset({'Coussin portrait'}), confidence=1.0, lift=13.230769230769232)]

reverse=True → from highest to lowest

## Plotting results
`enumerate(listRules)` gives each rule with its position `idx`, used to number the rules (Rule1, Rule2…).

In [13]:
for idx, rl in enumerate(listRules):
  print(f'\nRule{idx+1}')
  for statistic in rl:
    print(f'Item Base: {statistic.items_base}')
    print(f'Item Add: {statistic.items_add}')
    print(f'Confidence: {statistic.confidence}')
    print(f'Lift: {statistic.lift}')
    print('---------------------------------------------')


Rule1
Item Base: frozenset({'Coussin portrait'})
Item Add: frozenset({'Plaid portrait de votre animal'})
Confidence: 0.8461538461538463
Lift: 13.230769230769232
---------------------------------------------
Item Base: frozenset({'Plaid portrait de votre animal'})
Item Add: frozenset({'Coussin portrait'})
Confidence: 1.0
Lift: 13.230769230769232
---------------------------------------------

Rule2
Item Base: frozenset({'Collier cuir brodé au prénom'})
Item Add: frozenset({'Médaille ronde en laiton'})
Confidence: 0.6666666666666666
Lift: 2.293333333333333
---------------------------------------------
Item Base: frozenset({'Médaille ronde en laiton'})
Item Add: frozenset({'Collier cuir brodé au prénom'})
Confidence: 0.31999999999999995
Lift: 2.293333333333333
---------------------------------------------

Rule3
Item Base: frozenset({'Gamelle inox gravée'})
Item Add: frozenset({'Médaille ronde en laiton'})
Confidence: 0.625
Lift: 2.15
---------------------------------------------

Rule4
I

# 2. Check on the lecture examples
The same functions give the numbers of the lecture *Association Rule Mining*.

**Figure 1**: $sup(\{milk, bread\}) = 0.4$ and $conf(\{milk, bread\} \Rightarrow \{butter\}) = 0.5$ (with apyori, like the course).

In [14]:
figure_1 = [['milk', 'bread'], ['bread', 'butter'], ['beer'], ['milk', 'bread', 'butter'], ['bread', 'butter']]

for record in apriori(figure_1, min_support=0.2, min_confidence=0.0, min_lift=0.0):
    if record.items == {'milk', 'bread'}:
        print('sup({milk, bread}) =', record.support)
    if record.items == {'milk', 'bread', 'butter'}:
        for statistic in record.ordered_statistics:
            if statistic.items_base == {'milk', 'bread'}:
                print('conf({milk, bread} => {butter}) =', statistic.confidence)

sup({milk, bread}) = 0.4
conf({milk, bread} => {butter}) = 0.5


**Figure 2** (minsup 30 %, minconf 80 %): $\{chicken, clothes, milk\}$ is frequent with support 0.43 (3/7), and $\{clothes, chicken\} \Rightarrow \{milk\}$ has support 0.43 and confidence 1.

In [15]:
figure_2 = [['Beef', 'Chicken', 'Milk'], ['Beef', 'Cheese'], ['Cheese', 'Boots'], ['Beef', 'Chicken', 'Cheese'],
            ['Beef', 'Chicken', 'Clothes', 'Cheese', 'Milk'], ['Chicken', 'Clothes', 'Milk'], ['Chicken', 'Milk', 'Clothes']]

frequent = apriori_lecture(figure_2, minsup=0.3)
print('sup({Chicken, Clothes, Milk}) =', round(frequent[('Chicken', 'Clothes', 'Milk')], 2))
[rule for rule in generate_rules(frequent, minconf=0.8) if rule['A'] == ('Chicken', 'Clothes')]

sup({Chicken, Clothes, Milk}) = 0.43


[{'A': ('Chicken', 'Clothes'),
  'B': ('Milk',),
  'support': 0.42857142857142855,
  'confidence': 1.0,
  'lift': 1.7500000000000002}]

**Figure 3** (minsup 0.5): the iterations of Apriori. The lecture leaves $F_3$ as an exercise ("?").

In [16]:
figure_3 = [[1, 3, 4], [2, 3, 5], [1, 2, 3, 5], [2, 5]]   # T100, T200, T300, T400
frequent = apriori_lecture(figure_3, minsup=0.5, verbose=True)

k = 1   F1 = [(1,), (2,), (3,), (5,)]
k = 2   C2 = [(1, 2), (1, 3), (1, 5), (2, 3), (2, 5), (3, 5)]
        F2 = [(1, 3), (2, 3), (2, 5), (3, 5)]
k = 3   C3 = [(2, 3, 5)]
        F3 = [(2, 3, 5)]


**Example on candidate-gen**: from $F_3$, the join gives $\{1,2,3,4\}$ and $\{1,3,4,5\}$; $\{1,3,4,5\}$ is pruned because its subset $\{1,4,5\}$ is not frequent.

In [17]:
candidate_gen([(1, 2, 3), (1, 2, 4), (1, 3, 4), (1, 3, 5), (2, 3, 4)])

[(1, 2, 3, 4)]

# 3. Apriori step by step on our orders
`apriori_lecture()` follows the pseudo-code of the lecture (Figures 4 and 5). On our orders it finds **exactly the same frequent itemsets** as apyori (the library used by the course and the website).

In [18]:
frequent_lecture = apriori_lecture(records, minsup=MIN_SUPPORT, verbose=True)

k = 1   F1 = 16 itemsets
k = 2   C2 = 120 itemsets
        F2 = 9 itemsets
k = 3   C3 = [('Collier cuir brodé au prénom', 'Gamelle inox gravée', 'Médaille ronde en laiton'), ('Mug « Maman de… »', 'Mug « Papa de… »', 'Médaille ronde en laiton')]
        F3 = [('Collier cuir brodé au prénom', 'Gamelle inox gravée', 'Médaille ronde en laiton')]


In [19]:
# apyori with minconf = 0 and lift = 0 returns every frequent itemset
frequent_apyori = {frozenset(r.items): r.support for r in apriori(records, min_support=MIN_SUPPORT, min_confidence=0.0, min_lift=0.0)}
same = {frozenset(itemset) for itemset in frequent_lecture} == set(frequent_apyori)
print(len(frequent_lecture), 'frequent itemsets with the lecture pseudo-code,', len(frequent_apyori), 'with apyori. Same itemsets:', same)

26 frequent itemsets with the lecture pseudo-code, 26 with apyori. Same itemsets: True


# 4. Product page: *Fréquemment achetés ensemble*
Like the Amazon example of the lecture *Recommender Systems* (1.1, "Customers who bought this also bought"): on the page of product A we show the products B of the rules $\{A\} \Rightarrow \{B\}$, best confidence first.

`rules_to_json()` turns the results into simple data for `pages/product.php`.

In [20]:
data = rules_to_json(association_results, product_ids_by_name(order_items), len(records))

pd.set_option('display.width', 250)
pd.DataFrame([{'if (antecedent)': ' + '.join(r['if_names']), 'then (consequent)': ' + '.join(r['then_names']),
               'support': r['support'], 'confidence': r['confidence'], 'lift': r['lift']}
              for r in data['rules']])

,if (antecedent),then (consequent),support,confidence,lift
0,Plaid portrait de votre animal,Coussin portrait,0.0640,1.0000,13.2308
1,Gamelle inox gravée + Médaille ronde en laiton,Collier cuir brodé au prénom,0.0291,1.0000,7.1667
2,Collier cuir brodé au prénom + Gamelle inox gr...,Médaille ronde en laiton,0.0291,1.0000,3.4400
3,Coussin portrait,Plaid portrait de votre animal,0.0640,0.8462,13.2308
4,Médaille poisson pour chat,Collier chat avec grelot gravé,0.0465,0.7273,8.3394
5,Collier cuir brodé au prénom,Médaille ronde en laiton,0.0930,0.6667,2.2933
6,Gamelle inox gravée,Collier cuir brodé au prénom + Médaille ronde ...,0.0291,0.6250,6.7188
7,Gamelle inox gravée,Collier cuir brodé au prénom,0.0291,0.6250,4.4792
8,Gamelle inox gravée,Médaille ronde en laiton,0.0291,0.6250,2.1500
9,Médaille cœur émaillée,Gamelle céramique prénom,0.0349,0.5455,5.8636


# 5. Account page: *Recommandé pour vous*
Lecture *Recommender Systems*, **1.7.2 Generating Recommendations from Association Rules** (collaborative filtering):
1. ARM does not work on continuous values: we need a **binary user-item matrix**. The lecture converts ratings to 1/0; we have no ratings but purchases, which are already *implicit* 1s (1 = bought at least once, 0 = never).
2. Apriori is applied to this matrix (**1 transaction = 1 customer**). **The transactions of the target customer are ignored** during this step.
3. If the customer bought the left side of a rule, we recommend the right side (if not bought yet).

Thresholds: `USER_MIN_SUPPORT = 0.10` (at least ~5 of our ~50 customers), `USER_MIN_CONFIDENCE = 0.5`, `USER_MIN_LIFT = 1.2`.

In [21]:
matrix = user_item_matrix(order_items)   # 1 row per customer ('guest-…' = order without account)
matrix.head(8)

,Médaille os gravée,Collier cuir brodé au prénom,Gamelle céramique prénom,Plaid portrait de votre animal,Médaille ronde en laiton,Médaille cœur émaillée,Médaille poisson pour chat,Collier chat avec grelot gravé,Collier nylon coloris pastel,Harnais brodé au prénom,...,Set 2 gamelles + tapis,Coussin portrait,Couverture brodée au prénom,Mug « Maman de… »,Mug « Papa de… »,Gourde de balade gravée,Coffret adoption chiot,Coffret adoption chaton,Coffret anniversaire,Coffret « Moi & mon compagnon »
customer,,,,,,,,,,,,,,,,,,,,,
10,0,0,1,1,1,1,1,1,0,0,...,0,1,0,1,1,1,0,1,0,0
11,0,0,0,1,1,0,1,1,0,0,...,0,1,0,0,0,0,0,0,1,0
12,0,1,1,0,1,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0
13,0,0,1,0,1,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0
14,0,0,1,0,0,1,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
15,0,1,1,0,1,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
16,0,1,0,1,1,0,0,0,1,0,...,0,1,0,0,1,1,1,0,0,0
17,0,1,1,0,1,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


**The lecture example**: Alice liked Item1. Without Alice's row, Apriori finds $\{Item1\} \Rightarrow \{Item5\}$ with support 0.5 and confidence 1, so Item5 is recommended to Alice.

In [22]:
lecture_matrix = pd.DataFrame(
    [[1, 0, 0, 0, 0], [1, 0, 1, 0, 1], [1, 0, 1, 0, 1], [0, 0, 0, 1, 1], [0, 1, 1, 0, 0]],
    index=['Alice', 'User1', 'User2', 'User3', 'User4'], columns=['Item1', 'Item2', 'Item3', 'Item4', 'Item5'])
lecture_items = pd.DataFrame([(customer, int(item[-1]), item) for customer, row in lecture_matrix.iterrows()
                              for item in lecture_matrix.columns if row[item] == 1],
                             columns=['customer', 'product_id', 'product_name'])
pd.DataFrame(recommend_for_customers(lecture_items, min_support=0.5, min_confidence=0.5, min_lift=1.0)['Alice'])

,product_id,name,because_ids,because_names,support,confidence,lift
0,3,Item3,[1],[Item1],0.5,1.0,1.3333
1,5,Item5,[1],[Item1],0.5,1.0,1.3333


**Our customers**: recommendations for every customer with an account. Example: customer 19 (`zoe.renard`, password `Client123!`).

In [23]:
data['user_parameters'] = {'min_support': USER_MIN_SUPPORT, 'min_confidence': USER_MIN_CONFIDENCE, 'min_lift': USER_MIN_LIFT}
data['user_recommendations'] = recommend_for_customers(order_items)
print(len(data['user_recommendations']), 'customers get recommendations')
pd.DataFrame(data['user_recommendations'].get('19', []))[['name', 'because_names', 'support', 'confidence', 'lift']]

26 customers get recommendations


,name,because_names,support,confidence,lift
0,Médaille ronde en laiton,[Plaid portrait de votre animal],0.1346,0.7000,1.4000
1,Mug « Papa de… »,[Mug « Maman de… »],0.1731,0.6000,1.6421
2,Médaille cœur émaillée,[Gamelle céramique prénom],0.1346,0.5385,3.1111


Customers without any order get nothing: this is the **cold-start problem** of the lecture (1.5).

# 6. Evaluation: precision and recall
Lecture *Recommender Systems* **1.10**. For each customer with at least 2 orders, we hide the **last order**, recommend 4 products from the earlier orders only, and compare with what the customer really bought in that last order:

- $Precision = \frac{tp}{tp + fp}$ = good recommendations / all recommendations
- $Recall = \frac{tp}{tp + fn}$ = good recommendations / all products actually bought (that were new for the customer)

As the lecture says, these numbers do not fully predict the real behaviour of users online.

In [24]:
data['evaluation'] = evaluate_recommendations(order_items)
data['evaluation']

{'customers_evaluated': 19,
 'recommendations_per_customer': 4,
 'true_positives': 5,
 'false_positives': 22,
 'false_negatives': 18,
 'precision': 0.1852,
 'recall': 0.2174}

# Save the results for the website
`output/apriori_rules.json` is read by `pages/product.php` (part 4) and `pages/account.php` (part 5).

In [25]:
save_json(data)

Saved 16 rules (9 products, 26 customers) in analytics/output/apriori_rules.json
